# pyaegean Stage D+ — the combined retrain

Trains the full joint model on **AGDT + Gorman + Pedalion** (1.41M audited tokens).

**Targets (UD Perseus test):** LAS ≥ 84.0 (UAS already cleared its bar in both 9-epoch
runs); on PROIEL, lemma > 90.38.

**There is exactly ONE training cell (cell 4), controlled by the `EPOCHS` variable** —
the earlier two-cell design let a second run silently overwrite the first's checkpoint.
Set `EPOCHS`, run top to bottom, done.

In [ ]:
# 1 — confirm the GPU
import torch
print(torch.cuda.get_device_name(0))
print("bf16:", torch.cuda.is_bf16_supported())

In [ ]:
# 2 — install the package + a fresh clone
!pip install -q "git+https://github.com/ryanpavlicek/pyaegean" torch transformers numpy
!rm -rf pyaegean_repo
!git clone https://github.com/ryanpavlicek/pyaegean.git pyaegean_repo

In [ ]:
# 3 — build the combined dataset (sanity: gorman kept 26401 / excluded 1591;
#     train_tokens 1414213)
!python pyaegean_repo/training/build_full_dataset.py --with-extras

In [ ]:
# 4 — THE training cell (the only one). Current attempt: 12 epochs.
#     Sanity when done: the last epoch line says "epoch 11:", and metrics.json
#     will say "epochs": 12 (wall ~830-850s on the G4).
EPOCHS = 12
!python pyaegean_repo/training/train_full.py --model bowphs/GreBerta --epochs {EPOCHS}

**5 — note the best epoch's `lemma(best=<mode>)`** from cell 4's output — cell 6 needs
it. (If you want a different budget, change `EPOCHS` in cell 4 and re-run **that same
cell** — never add a second training cell; it would overwrite this checkpoint.)

In [ ]:
# 6 — headline measurements: set COMPOSE from cell 4's lemma(best=...) mode.
#     The output now echoes checkpoint_epochs — confirm it says 12.
COMPOSE = "lookup-first"  # <-- replace if cell 4 preferred another mode
!python pyaegean_repo/training/eval_full_ud.py \
    --checkpoint pyaegean_repo/training/out/full/model --compose {COMPOSE} \
    --treebank perseus --split test --out pyaegean_repo/training/out/full/ud-perseus-test.json
!python pyaegean_repo/training/eval_full_ud.py \
    --checkpoint pyaegean_repo/training/out/full/model --compose {COMPOSE} \
    --treebank proiel --split test --out pyaegean_repo/training/out/full/ud-proiel-test.json

In [ ]:
# 7 — save everything to Drive BEFORE the runtime recycles
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p "/content/drive/MyDrive/pyaegean-stage-dplus-12"
!cp -r pyaegean_repo/training/out/full "/content/drive/MyDrive/pyaegean-stage-dplus-12/"
!cp pyaegean_repo/training/data/full-stats.json "/content/drive/MyDrive/pyaegean-stage-dplus-12/"
!ls -la "/content/drive/MyDrive/pyaegean-stage-dplus-12/full" "/content/drive/MyDrive/pyaegean-stage-dplus-12/full/model"

## What to bring back

`full/metrics.json` (must say `"epochs": 12`), `full/ud-perseus-test.json`, and
`full/ud-proiel-test.json` (each now echoes `checkpoint_epochs: 12` — the overwrite
guard). The bars: **LAS ≥ 84.0** on Perseus; **lemma > 90.38** on PROIEL. The
checkpoint stays in Drive — whichever run wins becomes the Stage E export artifact.